In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import re

In [2]:
raw_path = Path.cwd().resolve().parent / 'data'/ 'raw'

In [3]:
snapshot_folder = raw_path / "2026-06-23"

calendar_path = snapshot_folder / "calendar.csv.gz"
listings_path = snapshot_folder / "listings.csv.gz"
reviews_path = snapshot_folder / "reviews.csv.gz"

calendar = pd.read_csv(calendar_path, encoding='utf-8', dtype={"listing_id": str})

listings = pd.read_csv(listings_path, encoding='utf-8', dtype={"id": str, "host_id": str, "listing_id": str})

reviews = pd.read_csv(reviews_path, encoding='utf-8', dtype={"id": str, "listing_id": str, "reviewer_id": str, "review_id": str})

In [4]:
print(len(calendar))
print(len(reviews))
print(reviews["Mẫu LLM"].notna().sum())
print(reviews["Nhãn tay"].notna().sum())

365000
5000
1000
100


In [5]:
def id_flags(data, prefix):
    """Tạo một cột True/False cho mỗi quy tắc kiểm tra ID."""
    ids = data['id']
    valid_id = ids.notna()
    repeated_id = valid_id & ids.duplicated()
    distinct_rows = data.loc[valid_id].drop_duplicates()
    conflicts = distinct_rows['id'].duplicated(keep=False)
    conflict_ids = distinct_rows.loc[conflicts, 'id']
    return pd.DataFrame({
        f'{prefix}_id_invalid': ~valid_id,
        f'{prefix}_duplicate_within': repeated_id,
        f'{prefix}_duplicate_conflict': ids.isin(conflict_ids),
    })

In [6]:
def clean_reviews(rv: pd.DataFrame, listings: pd.DataFrame):
    """Gắn vùng của listing vào review rồi kiểm tra ID và ngày."""
    before = rv.reset_index(drop=True).copy()
    before["comments"] = before["comments"].str.strip().str.replace(r"\r?<br\s*/?>", " ", regex=True, flags=re.I)
    before['review_date'] = pd.to_datetime(before["date"], format='%Y-%m-%d', errors='coerce')

    flags = id_flags(before, 'review')
    flags['review_date_missing'] = before["date"].isna()
    flags['review_date_invalid'] = before["date"].notna() & before['review_date'].isna()
    flags['review_outside_listings'] = ~(before['listing_id'].isin(listings["id"]))
    return before, flags

In [7]:
cleaned_rv, flags = clean_reviews(reviews, listings) 

In [8]:
rv_to_label = cleaned_rv[cleaned_rv["Nhãn tay"].str.strip().str.lower()=='x'][["id", "comments"]].reset_index(drop=True)
samples_path = Path.cwd().resolve().parent / 'data'/ 'samples'
samples_path.mkdir(parents=True, exist_ok=True)

In [12]:
nhan_tay = pd.read_csv(samples_path / "rv_to_label.csv", encoding='utf-8', dtype={"id": str})
nhan_tay.loc[rv_to_label["comments"] != nhan_tay["comments"]]

,id,comments,general_sentiment,accessibility,noise


In [ ]:
import sys
import yaml
from pathlib import Path

project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from src import llm

config_path = Path.cwd().resolve().parent / "configs" / "porto.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

In [ ]:
nhan_tay = pd.read_csv(samples_path / "rv_to_label.csv", encoding='utf-8')
labeled = llm.llm(nhan_tay, config_path)

In [ ]:
llm = labeled.add_prefix("llm_")
llm = llm.rename(columns={"llm_comments": "comments"})
so_sanh = nhan_tay.merge(llm, how='left', on="comments")
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].sum()

In [ ]:
so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]]

In [ ]:
so_sanh[so_sanh["noise"] != so_sanh["llm_noise"]]

In [ ]:
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]]

In [ ]:
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].iloc[0, 1]